In [1]:
"""
CLIP-ReID — универсальный инференс
====================================
Автоматически выбирает лучший бэкенд:
  TensorRT  →  ONNX CUDA  →  ONNX CPU
 
Зависимости:
  pip install torch torchvision onnx onnxruntime-gpu pillow numpy
  pip install git+https://github.com/openai/CLIP.git
  pip install tensorrt pycuda   # опционально, только для TRT
"""
 
import os
import time
import numpy as np
from PIL import Image
from typing import List, Optional, Tuple
import torchvision.transforms as T
import numpy as np

In [2]:
TRANSFORM = T.Compose([
    T.Resize((256, 128), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(
        mean=[0.48145466, 0.4578275,  0.40821073],
        std= [0.26862954, 0.26130258, 0.27577711],
    )
])

In [3]:
def preprocess(path: str) -> np.ndarray:
    """Изображение → np.ndarray [1, 3, 256, 128] float32."""
    return TRANSFORM(Image.open(path).convert("RGB")).unsqueeze(0)
 
def preprocess_batch(paths: List[str]) -> np.ndarray:
    """Список изображений → np.ndarray [N, 3, 256, 128] float32."""
    # return np.concatenate([preprocess(p) for p in paths])
    return np.array([preprocess(p) for p in paths])
 
# ──────────────────────────────────────────────────────────────────────────────
# ШАГ 1: ЭКСПОРТ .pth → .onnx
# Требует: репозиторий CLIP-ReID в PYTHONPATH, torch, clip
# ──────────────────────────────────────────────────────────────────────────────
 
def build_onnx(config_path: str,
               weights_path: str,
               onnx_path: str,
               opset: int = 14) -> None:
    """
    Конвертирует обученную модель CLIP-ReID (.pth) в ONNX формат.
 
    Запускать из корня репозитория Syliz517/CLIP-ReID, иначе
    импорты config и model.make_model_clipreid не найдутся.
 
    Args:
        config_path:  путь к .yml конфигу, например "configs/person/vit_clipreid.yml"
        weights_path: путь к .pth файлу,  например "MSMT17_clipreid_ViT-B-16_60.pth"
        onnx_path:    куда сохранить .onnx файл
        opset:        версия ONNX opset (14 — хорошая совместимость с TRT)
 
    Пример:
        build_onnx(
            config_path="configs/person/vit_clipreid.yml",
            weights_path="MSMT17_clipreid_ViT-B-16_60.pth",
            onnx_path="model.onnx",
        )
    """
    import torch
    import torch.nn as nn
    from config import cfg
    from model.make_model_clipreid import make_model
 
    # Загружаем модель на CPU — ONNX экспорт всегда на CPU, не вызываем .cuda()
    print(f"Загружаем модель: {weights_path}")
    cfg.merge_from_file(config_path)
    cfg.freeze()
 
    model = make_model(cfg, num_class=1041, camera_num=15, view_num=0)
    model.load_param(weights_path)
    model.eval()
    model.cpu()
 
    # Обёртка: фиксируем cam_label=0 и view_label=0 как константы внутри графа.
    # ONNX не поддерживает динамические управляющие аргументы снаружи.
    # Также приводим к float32 и L2-нормализуем выход.
    class _Wrapper(nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m
 
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            b    = x.shape[0]
            cam  = torch.zeros(b, dtype=torch.long)
            view = torch.zeros(b, dtype=torch.long)
            feat = self.m(x, get_image=True)
            if isinstance(feat, (tuple, list)):
                feat = feat[-1]
            feat = feat.float()
            return feat / feat.norm(dim=-1, keepdim=True).clamp(min=1e-12)  # [B, 512]
 
    wrapper = _Wrapper(model)
    wrapper.eval()
 
    print(f"Экспортируем в ONNX: {onnx_path}")
    dummy = torch.randn(1, 3, 256, 128)  # CPU
 
    torch.onnx.export(
        wrapper, dummy, onnx_path,
        input_names=["image"],
        output_names=["embedding"],
        dynamic_axes={"image": {0: "batch"}, "embedding": {0: "batch"}},
        opset_version=opset,
        do_constant_folding=True,
        training=torch.onnx.TrainingMode.EVAL,
        verbose=False,
    )
 
    # Валидация графа
    import onnx
    onnx.checker.check_model(onnx.load(onnx_path))
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"✓ ONNX сохранён: {onnx_path} ({size_mb:.1f} MB)")


In [4]:
def build_trt_engine(onnx_path: str,
                     engine_path: str,
                     precision: str = "fp16",
                     min_batch: int = 1,
                     opt_batch: int = 64,
                     max_batch: int = 128) -> None:
    """
    Конвертирует ONNX → TensorRT engine.
 
    Args:
        onnx_path:   путь к .onnx файлу
        engine_path: путь для сохранения .trt файла
        precision:   "fp32" | "fp16" (рекомендуется) | "int8"
                     fp16 — в 2x быстрее fp32, потеря качества < 0.1% mAP
        min_batch:   минимальный batch size
        opt_batch:   оптимальный batch size (TRT оптимизирует под него)
        max_batch:   максимальный batch size
 
    Пример:
        build_trt_engine("model.onnx", "model.trt", precision="fp16")
    """
    import tensorrt as trt
 
    logger  = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()
 
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)
 
    if precision == "fp16":
        if builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
            print("Режим: FP16")
        else:
            print("⚠ FP16 не поддерживается GPU, используем FP32")
    elif precision == "int8":
        if builder.platform_has_fast_int8:
            config.set_flag(trt.BuilderFlag.INT8)
            print("Режим: INT8")
        else:
            print("⚠ INT8 не поддерживается, используем FP16")
            config.set_flag(trt.BuilderFlag.FP16)
    else:
        print("Режим: FP32")
 
    print(f"Парсим ONNX: {onnx_path}")
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
            raise RuntimeError("Ошибка парсинга ONNX:\n" + "\n".join(errors))
 
    # Dynamic shapes — поддержка разных batch size при инференсе
    profile = builder.create_optimization_profile()
    profile.set_shape("image",
                      (min_batch, 3, 256, 128),   # min
                      (opt_batch, 3, 256, 128),   # opt — под него оптимизируем
                      (max_batch, 3, 256, 128))   # max
    config.add_optimization_profile(profile)
 
    print("Собираем engine (1–5 минут)...")
    t0 = time.time()
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("Не удалось собрать TensorRT engine")
 
    with open(engine_path, "wb") as f:
        f.write(serialized)
 
    size_mb = os.path.getsize(engine_path) / 1024 / 1024
    print(f"✓ Engine сохранён: {engine_path} ({size_mb:.1f} MB, {time.time()-t0:.0f} сек)")

In [5]:
class _ONNXBackend:
    def __init__(self, onnx_path: str, use_cuda: bool):
        import onnxruntime as ort
        providers = (["CUDAExecutionProvider"]
                     if use_cuda else ["CPUExecutionProvider"])
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        self.name = self.session.get_providers()[0]
 
    def run(self, x: np.ndarray) -> np.ndarray:
        return self.session.run(["embedding"], {"image": x})[0]
 
    def close(self): pass

In [6]:
class _TRTBackend:
    def __init__(self, engine_path: str):
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
 
        self._cuda = cuda
        runtime = trt.Runtime(trt.Logger(trt.Logger.WARNING))
 
        with open(engine_path, "rb") as f:
            engine = runtime.deserialize_cuda_engine(f.read())
 
        self.context  = engine.create_execution_context()
        self.stream   = cuda.Stream()
        self.inputs, self.outputs, self.bindings = [], [], []
        
        for i in range(engine.num_io_tensors):
            name      = engine.get_tensor_name(i)
            dtype     = trt.nptype(engine.get_tensor_dtype(name))
            raw_shape = engine.get_tensor_profile_shape(name, 0)[2]
            
            max_shape = [int(d) for d in raw_shape]
            
            size = 1
            for d in max_shape:
                size *= d
                
            host      = cuda.pagelocked_empty(size, dtype)
            device    = cuda.mem_alloc(host.nbytes)
            
            self.bindings.append(int(device))
            buf = {"name": name, "host": host, "device": device}
            (self.inputs if engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT
             else self.outputs).append(buf)
 
        self.name = "TensorRT"
 
    def run(self, x: np.ndarray) -> np.ndarray:
        cuda = self._cuda
        self.context.set_input_shape("image", x.shape)
        np.copyto(self.inputs[0]["host"][:x.size],
                  x.ravel())
        cuda.memcpy_htod_async(self.inputs[0]["device"],
                               self.inputs[0]["host"],
                               self.stream)
        
        self.context.execute_async_v3(stream_handle=self.stream.handle)
        
        cuda.memcpy_dtoh_async(self.outputs[0]["host"],
                               self.outputs[0]["device"],
                               self.stream)
        
        self.stream.synchronize()
        return self.outputs[0]["host"][:x.shape[0] * 512].reshape(x.shape[0], 512).copy()
 
    def close(self):
        for buf in self.inputs + self.outputs:
            buf["device"].free()

In [7]:
class CLIPReID:
    """
    Универсальный инференс CLIP-ReID.
 
    Args:
        onnx_path:  путь к .onnx файлу
        trt_path:   путь к .trt файлу (опционально)
        backend:    "auto" | "tensorrt" | "onnx_cuda" | "onnx_cpu"
        batch_size: размер батча при массовом инференсе
 
    Пример:
        reid = CLIPReID("model.onnx", trt_path="model.trt")
        emb   = reid.get_embedding("person.jpg")
        score = reid.compare("q.jpg", "g.jpg")
        top5  = reid.search("q.jpg", gallery_paths)
        reid.close()
    """
 
    def __init__(self,
                 onnx_path: str,
                 trt_path: Optional[str] = None,
                 backend: str = "auto",
                 batch_size: int = 16):
 
        self.batch_size = batch_size
        selected = self._select(backend, onnx_path, trt_path)
 
        if selected == "tensorrt":
            self._b = _TRTBackend(trt_path)
        elif selected == "onnx_cuda":
            self._b = _ONNXBackend(onnx_path, use_cuda=True)
        else:
            self._b = _ONNXBackend(onnx_path, use_cuda=False)
 
        print(f"CLIPReID | backend: {selected} | provider: {self._b.name}")
 
    def get_embedding(self, image_path: str) -> np.ndarray:
        """Извлекает L2-нормализованный эмбеддинг одного изображения → [512]."""
        return self._b.run(preprocess(image_path))[0]
 
    def get_embeddings(self, image_paths: List[str]) -> np.ndarray:
        """Извлекает эмбеддинги списка изображений → [N, 512]."""
        results = []
        for i in range(0, len(image_paths), self.batch_size):
            batch = preprocess_batch(image_paths[i : i + self.batch_size])
            results.append(self._b.run(batch))
        return np.array(results)
 
    def compare(self, path1: str, path2: str) -> float:
        """Cosine similarity двух изображений. ~1.0 = один человек."""
        return float(np.dot(self.get_embedding(path1), self.get_embedding(path2)))
 
    def search(self,
               query_path: str,
               gallery_paths: List[str],
               gallery_embs: Optional[np.ndarray] = None,
               k: int = 5) -> List[Tuple[str, float]]:
        """
        Ищет top-K похожих людей в галерее.
        Передавай gallery_embs повторно чтобы не пересчитывать каждый раз.
        Возвращает [(path, score), ...] по убыванию score.
        """
        if gallery_embs is None:
            gallery_embs = self.get_embeddings(gallery_paths)
        scores  = gallery_embs @ self.get_embedding(query_path)
        top_idx = np.argsort(scores)[::-1][:k]
        return [(gallery_paths[i], float(scores[i])) for i in top_idx]
 
    @staticmethod
    def _select(backend: str, onnx_path: str, trt_path: Optional[str]) -> str:
        if backend != "auto":
            return backend
        try:
            import tensorrt, pycuda.driver
            if trt_path and os.path.exists(trt_path):
                return "tensorrt"
        except ImportError:
            pass
        try:
            import torch
            if torch.cuda.is_available():
                return "onnx_cuda"
        except ImportError:
            pass
        return "onnx_cpu"
 
    def close(self):        self._b.close()
    def __enter__(self):    return self
    def __exit__(self, *_): self.close()


In [91]:
build_onnx(
      config_path="config/vit_clipreid-custom.yml",
      weights_path="weights/MSMT17_clipreid_ViT-B-16_60.pth",
      onnx_path="model.onnx",
  )

Загружаем модель: weights/MSMT17_clipreid_ViT-B-16_60.pth
Resized position embedding: %s to %s torch.Size([197, 768]) torch.Size([129, 768])
Position embedding resize to height:16 width: 8
Loading pretrained model from weights/MSMT17_clipreid_ViT-B-16_60.pth
Экспортируем в ONNX: model.onnx
✓ ONNX сохранён: model.onnx (329.0 MB)


In [93]:
# build_trt_engine("model.onnx", "model_int8.trt", precision='int8')
build_trt_engine("model.onnx", "model_float16.trt", precision='int8')

Режим: INT8
Парсим ONNX: model.onnx
Собираем engine (1–5 минут)...
[03/24/2026-10:57:40] [TRT] [W] Calibrator is not being used. Users must provide dynamic range for all tensors that are not Int32 or Bool.
[03/24/2026-10:57:40] [TRT] [E] IBuilder::buildSerializedNetwork: Error Code 4: Internal Error (Calibration failure occurred with no scaling factors detected. This could be due to no int8 calibrator or insufficient custom scales for network layers. Please see int8 sample to setup calibration correctly.)


RuntimeError: Не удалось собрать TensorRT engine

In [95]:
import tensorrt as trt

precision = 'int8'
logger  = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
network = builder.create_network(
    1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
)
parser = trt.OnnxParser(network, logger)
config = builder.create_builder_config()

config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)

if precision == "int8":
    if builder.platform_has_fast_int8:
        config.set_flag(trt.BuilderFlag.INT8)
        print("Режим: INT8")
    else:
        print("⚠ INT8 не поддерживается, используем FP16")
        config.set_flag(trt.BuilderFlag.FP16)
else:
    print("Режим: FP32")

print(f"Парсим ONNX: {onnx_path}")
with open(onnx_path, "rb") as f:
    if not parser.parse(f.read()):
        errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
        raise RuntimeError("Ошибка парсинга ONNX:\n" + "\n".join(errors))

# Dynamic shapes — поддержка разных batch size при инференс

Режим: INT8
Парсим ONNX: model.onnx


In [97]:
profile = builder.create_optimization_profile()
profile.set_shape("image",
                  (16, 3, 256, 128),   # min
                  (64, 3, 256, 128),   # opt — под него оптимизируем
                  (128, 3, 256, 128))   # max
config.add_optimization_profile(profile)

print("Собираем engine (1–5 минут)...")
t0 = time.time()
serialized = builder.build_serialized_network(network, config)
if serialized is None:
    raise RuntimeError("Не удалось собрать TensorRT engine")

with open(engine_path, "wb") as f:
    f.write(serialized)

Собираем engine (1–5 минут)...
[03/24/2026-10:59:25] [TRT] [W] Calibrator is not being used. Users must provide dynamic range for all tensors that are not Int32 or Bool.
[03/24/2026-10:59:25] [TRT] [E] IBuilder::buildSerializedNetwork: Error Code 4: Internal Error (Calibration failure occurred with no scaling factors detected. This could be due to no int8 calibrator or insufficient custom scales for network layers. Please see int8 sample to setup calibration correctly.)


RuntimeError: Не удалось собрать TensorRT engine

In [98]:
builder.build_serialized_network(network, config)

[03/24/2026-10:59:34] [TRT] [W] Calibrator is not being used. Users must provide dynamic range for all tensors that are not Int32 or Bool.
[03/24/2026-10:59:34] [TRT] [E] IBuilder::buildSerializedNetwork: Error Code 4: Internal Error (Calibration failure occurred with no scaling factors detected. This could be due to no int8 calibrator or insufficient custom scales for network layers. Please see int8 sample to setup calibration correctly.)


In [43]:
reid = _ONNXBackend("model.onnx", use_cuda=True)

2026-03-18 13:51:49.211106938 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


In [8]:
reid = CLIPReID(onnx_path = 'model.onnx', trt_path='model_float16.trt',  backend='onnx_cuda')

CLIPReID | backend: onnx_cuda | provider: CUDAExecutionProvider


2026-04-01 06:01:58.770646245 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-04-01 06:01:58.776942649 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-04-01 06:01:58.776958927 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [9]:
reid

In [18]:
from pathlib import Path

In [20]:
images = list(Path('data/outputs/images').rglob('*.jpg'))

In [29]:
reid.get_embedding(images[0]).shape

(1280,)

In [22]:
reid.get_embeddings((Image.open(images[0])))

TypeError: object of type 'JpegImageFile' has no len()

In [25]:
build_trt_engine(
    onnx_path="model.onnx",
    engine_path="model.trt",
    opt_batch=8,
    max_batch=32)

Режим: FP16
Парсим ONNX: model.onnx
Собираем engine (1–5 минут)...
[03/12/2026-07:27:56] [TRT] [W] Detected layernorm nodes in FP16.
[03/12/2026-07:27:56] [TRT] [W] Running layernorm after self-attention in FP16 may cause overflow. Exporting the model to the latest available ONNX opset (later than opset 17) to use the INormalizationLayer, or forcing layernorm layers to run in FP32 precision can help with preserving accuracy.
✓ Engine сохранён: model.trt (169.0 MB, 24 сек)


In [10]:
from pathlib import Path
from PIL import Image

In [11]:
images = list(Path('data/outputs/images').rglob('*.jpg'))

In [17]:
reid = CLIPReID("model.onnx", trt_path="")

CLIPReID | backend: onnx_cuda | provider: CUDAExecutionProvider


2026-03-18 13:39:36.141251685 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


In [20]:
model = _ONNXBackend("model.onnx", use_cuda = True)

2026-03-18 13:43:18.942217191 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


In [49]:
model.session.run(['embedding'], {"image": preprocess(images[0])})

[array([-0.06460667,  0.08373655, -0.0303823 , ..., -0.01485891,
         0.0345729 ,  0.01423603], dtype=float32)]

In [48]:
verbose = False

similary = 0.8 # степень схожести
distance = 0.4 # дистанция по евклидовуму расстоянию

work_file = './final_video_file' # директория, где хранятся видео

for file in list(Path(work_file).rglob('*.mp4')):
    print(f"Файл в обработке {file}")
    stored_data = person_tracker.process_video(
        video_path = file,
        yolo_person_model=yolo_person,
        stored_data = stored_data,
        json_output_dir = os.path.dirname(CONFIG["output_json"]),

        excel_output_path = CONFIG["output_excel"],
        images_output_dir = CONFIG['output_images'],
        clip_reid=CLIP_reid,
        libs='clipreid-vit',
        
        distance_threshold = distance,
        similarity_threshold = similary,
        max_frame_gap = 400,
    )

    save_json(stored_data, CONFIG["output_json"])
    verbose = False

locations = OrderedDict(stored_data)
locations['parameters'] = ['similarity', similary, 'distance', distance]
locations.move_to_end('parameters',last=False)
locations['weight_model'] = 'clip'
locations.move_to_end('weight_model', last=False)
save_json(locations, CONFIG["output_json"])

(3, 256, 128)

In [ ]:
Image.open()

In [38]:
model.run({"image": preprocess(images[0])[0]})

Fail: <class 'TypeError'>: only length-1 arrays can be converted to Python scalars

In [22]:
g_embs.shape

(8, 1280)

In [45]:
image = TRANSFORM(Image.open(images[0])).unsqueeze(0).cuda()
# with torch.no_grad():
#     feat = model(image, get_image=True)

In [53]:
image = TRANSFORM(Image.open(images[0]).convert("RGB")).unsqueeze(0).numpy()